# Declarative visualization

> **Krisk 0.9 notebook mode**
>
> This notebook runs locally with pandas and Krisk only. It does not require the Krisk
> server, MCP, SQLite, or PostgreSQL. Chart output embeds its renderer and can be used
> offline. The typed `ChartSpec` API is recommended for new work; legacy `krisk.plot`
> calls remain supported for existing notebooks.

A declarative chart describes *what* to show rather than assembling ECharts
callbacks. `ChartSpec` is validated, JSON-safe, and portable across notebooks, REST,
MCP, saved snapshots, and research exports.

In [ ]:
from pathlib import Path

import pandas as pd
import krisk.plot as kk
from krisk import Chart, ChartSpec, __version__

ROOT = Path.cwd()
if not (ROOT / "krisk" / "tests" / "data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "krisk" / "tests" / "data"

print(f"Krisk {__version__}")

In [ ]:
sales = pd.DataFrame(
    {
        "month": ["Jan", "Feb", "Mar", "Apr"],
        "revenue": [82, 93, 90, 112],
        "channel": ["Direct", "Partner", "Direct", "Partner"],
    }
)

## A portable typed specification

In [ ]:
spec = ChartSpec(
    kind="bar",
    x="month",
    y="revenue",
    color="channel",
    aggregate="sum",
    title="Revenue by month",
    description="Recorded revenue grouped by acquisition channel",
    safe_overrides={
        "legend": {"top": 8},
        "grid": {"left": 48, "right": 24, "bottom": 44},
    },
)
chart = Chart.from_dataframe(sales, spec)
chart

The validated specification can be serialized without code execution.

In [ ]:
spec.model_dump(mode="json")

The compiled ECharts option is also available for inspection.

In [ ]:
chart.option

## Raw `Chart` compatibility

The historical `Chart()` constructor remains available for existing notebooks.
It is a low-level escape hatch and is not accepted as an MCP/REST chart specification.

In [ ]:
raw_chart = Chart()
raw_chart.set_title("A manually assembled chart", x_pos="center")
raw_chart.option["series"] = [
    {"data": [10, 3, 7, 4, 5], "name": "observations", "type": "bar"}
]
raw_chart.option["xAxis"] = {
    "data": ["Americas", "Asia", "Africa", "Oceania", "Europe"]
}
raw_chart

Use JSON-compatible values in `safe_overrides`. Callback-shaped keys and strings
containing JavaScript functions are rejected by `ChartSpec`; this keeps saved and
LLM-created charts declarative.